In [4]:
!pip install pandas
!pip install seaborn
!pip install matplotlib
!pip install statsmodels
!pip install scikit-learn

import pandas as pd  # data tables
import numpy as np  # math helpers
from sklearn.model_selection import train_test_split  # splits data into train/test
from sklearn.preprocessing import StandardScaler  # puts all features on the same scale
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV  # the models
from sklearn.metrics import r2_score, mean_squared_error  # ways to score the models

# ---------- DATSET #1: Predicting Insurance Prices ----------
# Analyzing medical insurance prices across regions  https://www.kaggle.com/code/kellibelcher/predicting-insurance-prices-across-regions/input
# Last week, I did some EDA on my datasets and did regular linear regression on all variables to predict medical insurance price. There were a good mix
# of categorical (region, smoker vs. non smoker, etc.) and continuous variables (age, BMI, etc.). I one-hot encoded the categorical features to make them
# numeric in order to run the regression on all terms.
# Based off my findings from my EDA when I ran correlation matrix heat map and scatterplots, I tested some interaction and quadratic terms.
# So I'm going to add in some useful interaction terms and quadratic term to see how the different regularization methods impact the model.

# ---------- 1. Rebuilding Week 1 features ----------

df = pd.read_csv("insurance.csv")  # load data
y = df['charges']  # target
X = pd.get_dummies(df.drop(columns='charges'), drop_first=True).astype(float)  # dummies for categorical columns
X['smoker_bmi'] = X['smoker_yes'] * df['bmi']  # interaction 1
X['smoker_age'] = X['smoker_yes'] * df['age']  # interaction 2 (let lasso decide if it matters!)
X['age_sq'] = df['age']**2  # polynomial term

# ---------- 2. Train/testing split ----------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 80% train, 20% test

# ---------- 3. Scaling the features (needed for selection methods) ---------- 
scaler = StandardScaler()  # create the scaler
X_train_s = scaler.fit_transform(X_train)  # learn the scale from training data, then apply it
X_test_s = scaler.transform(X_test)  # apply the same scale to test data

# ---------- 4. Fitting the models ----------
ols = LinearRegression().fit(X_train_s, y_train)  # plain regression, no penalty (for comparison)
ridge = RidgeCV(alphas=np.logspace(-2, 3, 50)).fit(X_train_s, y_train)  # ridge, tries 50 alphas and picks the best
lasso = LassoCV(cv=5, random_state=42, max_iter=10000).fit(X_train_s, y_train)  # lasso, picks best alpha using 5-fold CV
enet = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], cv=5, random_state=42, max_iter=10000).fit(X_train_s, y_train)  # elastic net

# I used 5-fold cross-validation to pick the penalty strength (alpha), which is a common default that works well for a dataset this size. 
# For ridge, I searched a wide range of alphas (0.01 to 1,000) because I didn't know the best value in advance. 
# For elastic net, I tested a few ridge/lasso mixes (0.1, 0.5, 0.9) to compare mostly-ridge, balanced, and mostly-lasso models.
print("Best alpha - Ridge:", ridge.alpha_)  # chosen penalty strength
print("Best alpha - Lasso:", lasso.alpha_)
print("Best alpha - Elastic Net:", enet.alpha_, "| l1_ratio:", enet.l1_ratio_)  # l1_ratio: 1 = all lasso, 0 = all ridge

# ---------- 5. Comparing performance ----------
models = {'OLS': ols, 'Ridge': ridge, 'Lasso': lasso, 'Elastic Net': enet} 
rows = []  
for name, m in models.items():  
    pred = m.predict(X_test_s)  
    rows.append({
        'Model': name,
        'Train R2': m.score(X_train_s, y_train),  # fit on data it has seen
        'Test R2': r2_score(y_test, pred),  # fit on data it has NOT seen
        'Test RMSE': np.sqrt(mean_squared_error(y_test, pred))  # average prediction error in dollars
    })
results = pd.DataFrame(rows).round(4)  # turn into a table
print("\nMODEL COMPARISON")
print(results)

# ---------- 6. Comparing coefficients ----------
coefs = pd.DataFrame({
    'OLS': ols.coef_, 'Ridge': ridge.coef_,
    'Lasso': lasso.coef_, 'Elastic Net': enet.coef_
}, index=X.columns).round(1)  
print("\nCOEFFICIENTS (scaled features)")
print(coefs) 

Best alpha - Ridge: 0.33932217718953295
Best alpha - Lasso: 10.08022686568896
Best alpha - Elastic Net: 11.200252072987732 | l1_ratio: 0.9

MODEL COMPARISON
         Model  Train R2  Test R2  Test RMSE
0          OLS    0.8368   0.8673  4538.3898
1        Ridge    0.8367   0.8675  4534.9748
2        Lasso    0.8366   0.8676  4533.2847
3  Elastic Net    0.7062   0.7355  6408.1263

COEFFICIENTS (scaled features)
                      OLS    Ridge    Lasso  Elastic Net
age                -542.9   -472.2     -0.0       1024.5
bmi                  74.9     97.8    115.6        844.2
children            754.2    750.4    719.0        299.0
sex_male           -263.5   -260.5   -246.1         58.3
smoker_yes        -8606.2  -8381.6  -8060.8       2244.3
region_northwest   -266.8   -265.1   -233.2        -54.5
region_southeast   -419.5   -417.5   -380.9         52.4
region_southwest   -508.9   -506.7   -474.8       -101.7
smoker_bmi        18590.9  18374.1  18093.0       2800.4
smoker_age      

The Trained and Test R2 between OLS, Ridge, and Lasso are basically the same models in accuracy. All are high.
Elastic Net had a substantially lower Train and Test R2 than the other models. Its still decently high, but at least 13% lower (in Test R2) than the other models.
Lasso deemed Age and the interaction term of smoker & age as insignificant variables in the model to predicting price.
If I was to choose one of these models, I would probably pick Lasso. This is because the R2 is tied for the highest, and it recommends dropping certain features.
I prefer simpler models in large part to avoid overfitting and any multicollinearity issues.

In [5]:
# ---------- DATSET #2: Insurance Fraud ----------
# Analyzing insurance claims https://data.mendeley.com/datasets/992mh7dk9y/2
# Last week, I did EDA on this dataset and ran regular linear regression on all variables to predict the total claim amount. There was a good mix
# of categorical (incident severity, collision type, etc.) and continuous variables (age, months as customer, policy premium, etc.). I replaced the "?" missing values
# with an "Unknown" category and one-hot encoded the categorical features to make them numeric in order to run the regression on all terms.
# Based off my findings from my EDA when I ran the correlation matrix heat map and scatterplots, I tested an interaction term (major damage x number of vehicles)
# and a quadratic term (age squared). Neither improved adjusted R² and neither was statistically significant, and age and months as customer were highly
# correlated (0.92), which showed up as elevated VIF. So this week I'm going to keep those terms in the model to see how the different regularization methods handle them. 

# ---------- 1. Rebuilding Week 1 features ----------
df = pd.read_csv("insurance_claims.csv")  # load data (change to your file name)
df = df.replace('?', 'Unknown')  # assuming "?" means missing

num_cols = ['age', 'months_as_customer', 'policy_annual_premium', 'number_of_vehicles_involved']  # continuous predictors
cat_cols = ['incident_severity', 'collision_type', 'insured_sex', 'property_damage']  # categorical predictors
df = df[num_cols + cat_cols + ['total_claim_amount']].dropna()  # keep only the columns we need

y = df['total_claim_amount']  # target
X = pd.get_dummies(df.drop(columns='total_claim_amount'), columns=cat_cols, drop_first=True).astype(float)  # dummies for categorical columns
major = (df['incident_severity'] == 'Major Damage').astype(float)  # temporary 1/0 flag (not added to X on its own)
X['major_x_vehicles'] = major * df['number_of_vehicles_involved']  # interaction term
X['age_sq'] = df['age']**2  # polynomial term

# ---------- 2. Train/testing split ----------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 80% train, 20% test

# ---------- 3. Scaling the features (needed for selection methods) ----------
scaler = StandardScaler() 
X_train_s = scaler.fit_transform(X_train) 
X_test_s = scaler.transform(X_test) 

# ---------- 4. Fitting the models ----------
alphas = np.logspace(-2, 3, 50)  # wide range of penalties (0.01 to 1000) so the best one isn't cut off
ols = LinearRegression().fit(X_train_s, y_train)  # plain regression, no penalty (for comparison)
ridge = RidgeCV(alphas=alphas, cv=5).fit(X_train_s, y_train)  # ridge, tries 100 alphas and picks the best
lasso = LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=10000).fit(X_train_s, y_train)  # lasso, picks best alpha using 5-fold CV
enet = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 0.95, 1], alphas=alphas, cv=5, random_state=42, max_iter=10000).fit(X_train_s, y_train)  # elastic net

# I used 5-fold cross-validation to pick the penalty strength (alpha), which is a common default that works well for a dataset this size.
# I searched a wide range of alphas (0.01 to 1,000) because I didn't know the best value in advance.
# For elastic net, I tested a few ridge/lasso mixes to compare mostly-ridge, balanced, and mostly-lasso models.
print("Best alpha - Ridge:", ridge.alpha_)
print("Best alpha - Lasso:", lasso.alpha_) 
print("Best alpha - Elastic Net:", enet.alpha_, "| l1_ratio:", enet.l1_ratio_)  # l1_ratio: 1 = all lasso, 0 = all ridge

# ---------- 5. Comparing performance ----------
models = {'OLS': ols, 'Ridge': ridge, 'Lasso': lasso, 'Elastic Net': enet}  
rows = [] 
for name, m in models.items():  
    pred = m.predict(X_test_s) 
    rows.append({
        'Model': name,
        'Train R2': m.score(X_train_s, y_train),  
        'Test R2': r2_score(y_test, pred), 
        'Test RMSE': np.sqrt(mean_squared_error(y_test, pred)) 
    })
results = pd.DataFrame(rows).round(4) 
print("\nMODEL COMPARISON")
print(results)

# ---------- 6. Comparing coefficients ----------
coefs = pd.DataFrame({
    'OLS': ols.coef_, 'Ridge': ridge.coef_,
    'Lasso': lasso.coef_, 'Elastic Net': enet.coef_
}, index=X.columns).round(1) 
print("\nCOEFFICIENTS (scaled features)")
print(coefs)

Best alpha - Ridge: 1.0985411419875584
Best alpha - Lasso: 390.6939937054621
Best alpha - Elastic Net: 390.6939937054621 | l1_ratio: 1.0

MODEL COMPARISON
         Model  Train R2  Test R2   Test RMSE
0          OLS    0.7070   0.6865  14444.1948
1        Ridge    0.7070   0.6878  14415.1987
2        Lasso    0.7036   0.6950  14246.0690
3  Elastic Net    0.7036   0.6950  14246.0690

COEFFICIENTS (scaled features)
                                      OLS    Ridge    Lasso  Elastic Net
age                               -5424.9  -4617.0      0.0          0.0
months_as_customer                 -308.9   -316.6      0.0          0.0
policy_annual_premium              -244.8   -248.8     -0.0         -0.0
number_of_vehicles_involved       -1501.5  -1485.8  -1035.7      -1035.7
incident_severity_Minor Damage     -727.6   -760.3     -0.0         -0.0
incident_severity_Total Loss      -1470.3  -1486.0   -456.5       -456.5
incident_severity_Trivial Damage   -332.8   -398.2     -0.0         -0.0

The OLS and Ridge models have basically the same Train and Test R2, while Lasso and Elastic Net both had the same Train and Test R2. OLS and Ridge score slightly higher on the training data, but Lasso and Elastic Net score slightly higher on the test data. This suggests the penalty helped reduce overfitting. 
The Lasso model shows that age, months_as_customer, policy_annual_premium, incident_severity_Minor Damage, incident_severity_Trivial Damage, incident_severity_Side Collision, insured_sex_MALE, property_damage_Unknown, and major_x_vehicles.